In [ ]:
import json
import subprocess
import tempfile
import os
import re
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple, Optional

In [ ]:
def download_from_gcs(gcs_path: str, local_path: str) -> None:
    """Download a file from GCS to local path."""
    subprocess.check_call(["gsutil", "-m", "cp", gcs_path, local_path])


def load_json_from_gcs(gcs_path: str) -> dict:
    """Load a JSON file from GCS."""
    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.json') as tmp_file:
        tmp_path = tmp_file.name
    
    try:
        download_from_gcs(gcs_path, tmp_path)
        with open(tmp_path, 'r') as f:
            data = json.load(f)
        return data
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)


def list_gcs_files(gcs_dir: str) -> List[str]:
    """List all files in a GCS directory."""
    result = subprocess.run(
        ["gsutil", "ls", gcs_dir],
        capture_output=True,
        text=True,
        check=True
    )
    return [line.strip() for line in result.stdout.split('\n') if line.strip() and line.strip().endswith('.json')]

In [ ]:
# C4 perplexity key in the JSON files
C4_KEY = "eval-data_perplexity_v3_small_gptneox20b_c4_en_val_part-0-00000"

# GCS base directory
GCS_BASE_DIR = "gs://cmu-gpucloud-catheri4/outputs/muon/ModelEvaluation"

# Specific file path provided by user
SPECIFIC_FILE = "gs://cmu-gpucloud-catheri4/outputs/muon/ModelEvaluation/OLMo-tk16B-muon-lr9e-3-wd1e-1-bs256-muon_lr5e-3-eval.json"

In [ ]:
def extract_tokens_from_filename(filename: str) -> Optional[int]:
    """Extract token count from filename like 'OLMo-tk16B-...'."""
    match = re.search(r'tk(\d+)B', filename)
    if match:
        return int(match.group(1))
    return None


def extract_group_info(filename: str) -> Dict[str, Optional[str]]:
    """Extract group information from filename."""
    info = {}
    
    # Extract tokens
    info['tokens'] = extract_tokens_from_filename(filename)
    
    # Extract learning rate (e.g., lr9e-3)
    lr_match = re.search(r'-lr([0-9eE\+\-\.]+)', filename)
    info['lr'] = lr_match.group(1) if lr_match else None
    
    # Extract weight decay (e.g., wd1e-1)
    wd_match = re.search(r'-wd([0-9eE\+\-\.]+)', filename)
    info['wd'] = wd_match.group(1) if wd_match else None
    
    # Extract batch size (e.g., bs256)
    bs_match = re.search(r'-bs(\d+)', filename)
    info['bs'] = int(bs_match.group(1)) if bs_match else None
    
    # Extract muon_lr (e.g., muon_lr5e-3)
    muon_lr_match = re.search(r'-muon_lr([0-9eE\+\-\.]+)', filename)
    info['muon_lr'] = muon_lr_match.group(1) if muon_lr_match else None
    
    # Create a group identifier based on hyperparameters (excluding tokens)
    group_parts = []
    if info['lr']:
        group_parts.append(f"lr{info['lr']}")
    if info['wd']:
        group_parts.append(f"wd{info['wd']}")
    if info['bs']:
        group_parts.append(f"bs{info['bs']}")
    if info['muon_lr']:
        group_parts.append(f"muon_lr{info['muon_lr']}")
    
    info['group'] = '-'.join(group_parts) if group_parts else 'default'
    
    return info

In [ ]:
# Load the specific file provided
print(f"Loading file: {SPECIFIC_FILE}")
data = load_json_from_gcs(SPECIFIC_FILE)
print(f"Keys in JSON: {list(data.keys())}")
print(f"\nC4 Perplexity: {data.get(C4_KEY, 'Not found')}")

In [ ]:
# Try to list all evaluation files in the directory to find multiple groups
try:
    print(f"Listing files in {GCS_BASE_DIR}...")
    all_files = list_gcs_files(GCS_BASE_DIR)
    print(f"Found {len(all_files)} JSON files")
    
    # Filter for eval.json files
    eval_files = [f for f in all_files if 'eval.json' in f]
    print(f"Found {len(eval_files)} eval.json files")
    
    if len(eval_files) > 0:
        print("\nSample files:")
        for f in eval_files[:5]:
            print(f"  {f}")
except Exception as e:
    print(f"Could not list directory (may need to specify files manually): {e}")
    eval_files = [SPECIFIC_FILE]

In [ ]:
# Collect data from all files
results = {}  # {group: {tokens: c4_perplexity}}

for gcs_file in eval_files:
    try:
        # Extract filename from path
        filename = os.path.basename(gcs_file)
        
        # Extract group info
        info = extract_group_info(filename)
        group = info['group']
        tokens = info['tokens']
        
        if tokens is None:
            print(f"Warning: Could not extract tokens from {filename}, skipping")
            continue
        
        # Load JSON and extract C4 perplexity
        data = load_json_from_gcs(gcs_file)
        c4_pplx = data.get(C4_KEY)
        
        if c4_pplx is None:
            print(f"Warning: C4 perplexity not found in {filename}, skipping")
            continue
        
        # Store result
        if group not in results:
            results[group] = {}
        results[group][tokens] = float(c4_pplx)
        
        print(f"✓ {filename}: group={group}, tokens={tokens}B, c4_pplx={c4_pplx:.4f}")
        
    except Exception as e:
        print(f"Error processing {gcs_file}: {e}")
        continue

print(f"\nCollected data for {len(results)} group(s)")
for group, data in results.items():
    print(f"  {group}: {len(data)} data points")


In [ ]:
# Plot C4 perplexity vs tokens for each group
fig, ax = plt.subplots(figsize=(10, 6))

# Color cycle for different groups
colors = plt.cm.tab10(np.linspace(0, 1, len(results)))
markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']

for idx, (group, data) in enumerate(sorted(results.items())):
    # Sort by tokens
    tokens_list = sorted(data.keys())
    c4_values = [data[t] for t in tokens_list]
    
    # Plot
    color = colors[idx % len(colors)]
    marker = markers[idx % len(markers)]
    ax.plot(tokens_list, c4_values, marker=marker, label=group, 
            color=color, linewidth=2, markersize=8)
    
    # Add text annotations
    for t, v in zip(tokens_list, c4_values):
        ax.annotate(f'{v:.3f}', (t, v), textcoords="offset points", 
                   xytext=(0,10), ha='center', fontsize=8)

ax.set_xlabel('Number of Tokens (B)', fontsize=12)
ax.set_ylabel('C4 Perplexity', fontsize=12)
ax.set_title('C4 Perplexity vs Number of Tokens', fontsize=14)
ax.legend(title='Group', fontsize=10, title_fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(sorted(set([t for group_data in results.values() for t in group_data.keys()])))

plt.tight_layout()
plt.show()